# Pronósticos con Prophet para Series Temporales

**Elaborado por:** David Palacio J.  
**Correo:** davidpalacioj@gmail.com

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dpalacioj/DataAI-Fundamentos-Aplicaciones/blob/main/Analitica/ciencia_datos_con_python/notebooks_teoria/08_2_prophet_series_temporales.ipynb)

---

## 🚀 ¿Qué es Prophet?

**Prophet** es una herramienta desarrollada por Meta (Facebook) para hacer **pronósticos de series temporales** de forma sencilla y eficiente.

### ✨ ¿Por qué Prophet es especial?

**🎯 Fácil de usar**: No necesitas ser experto en estadística
- Solo necesitas datos con fechas y valores
- El modelo se ajusta automáticamente
- Funciona bien "out of the box"

**🔧 Maneja automáticamente**:
- **Tendencias** que cambian con el tiempo
- **Estacionalidad** anual, semanal y diaria
- **Días festivos** y eventos especiales
- **Datos faltantes** sin problemas

**📊 Interpretabilidad**:
- Te dice qué componentes influyen más
- Gráficos fáciles de entender
- Intervalos de confianza automáticos

## 🛠️ Instalación

Para usar Prophet, primero necesitamos instalarlo. En Google Colab:

In [ ]:
# Instalar Prophet (solo necesario la primera vez)
!pip install prophet --quiet

# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

# Configuración para mejores gráficos
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
print("✅ Prophet instalado y librerías importadas")

## 📈 Ejemplo Práctico: Ventas de una Tienda Online

Vamos a simular datos de ventas diarias de una tienda online durante 3 años, incluyendo:
- **Tendencia creciente** (la tienda se vuelve más popular)
- **Estacionalidad anual** (más ventas en noviembre/diciembre)
- **Estacionalidad semanal** (menos ventas los lunes)
- **Ruido aleatorio** (variaciones impredecibles)

In [ ]:
# Crear datos sintéticos de ventas diarias
np.random.seed(42)

# Rango de fechas: 3 años de datos diarios
dates = pd.date_range(start='2021-01-01', end='2023-12-31', freq='D')
n_days = len(dates)

# Componentes de la serie temporal
# 1. Tendencia creciente
trend = np.linspace(100, 300, n_days)

# 2. Estacionalidad anual (pico en Black Friday/Navidad)
day_of_year = dates.dayofyear
annual_seasonality = 50 * np.sin(2 * np.pi * (day_of_year - 60) / 365) + \
                    30 * np.sin(2 * np.pi * (day_of_year - 330) / 365)

# 3. Estacionalidad semanal (menos ventas lunes, más viernes/sábado)
day_of_week = dates.dayofweek  # 0=Lunes, 6=Domingo
weekly_pattern = [-20, -10, 5, 10, 25, 30, 15]  # Lun a Dom
weekly_seasonality = [weekly_pattern[dow] for dow in day_of_week]

# 4. Ruido aleatorio
noise = np.random.normal(0, 15, n_days)

# Combinar todos los componentes
sales = trend + annual_seasonality + weekly_seasonality + noise
sales = np.maximum(sales, 0)  # Las ventas no pueden ser negativas

# Crear DataFrame en formato que Prophet espera
df = pd.DataFrame({
    'ds': dates,  # Prophet requiere que la columna de fechas se llame 'ds'
    'y': sales    # Prophet requiere que la columna de valores se llame 'y'
})

print(f"✅ Creados {len(df)} días de datos de ventas")
print(f"📊 Rango de fechas: {df['ds'].min().date()} a {df['ds'].max().date()}")
print(f"💰 Ventas promedio: ${df['y'].mean():.0f}")
print("\nPrimeros 10 días:")
print(df.head(10))

### 📊 Visualicemos los datos históricos

In [ ]:
# Graficar la serie temporal completa
plt.figure(figsize=(15, 6))
plt.plot(df['ds'], df['y'], alpha=0.8, linewidth=1)
plt.title('Ventas Diarias de Tienda Online (2021-2023)', fontsize=16)
plt.xlabel('Fecha')
plt.ylabel('Ventas ($)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("👀 ¿Qué patrones puedes observar?")
print("📈 Tendencia: ¿Las ventas aumentan con el tiempo?")
print("🔄 Estacionalidad: ¿Hay picos regulares cada año?")
print("📅 Patrones semanales: ¿Ciertos días venden más?")

## 🎯 Entrenando el Modelo Prophet

Prophet es increíblemente fácil de usar. Solo necesitamos:
1. Crear el modelo
2. Entrenarlo con los datos históricos
3. Hacer predicciones para el futuro

In [ ]:
# 1. Crear el modelo Prophet
model = Prophet(
    yearly_seasonality=True,    # Detectar patrones anuales
    weekly_seasonality=True,    # Detectar patrones semanales
    daily_seasonality=False,    # No necesitamos patrones por hora
    interval_width=0.95         # Intervalo de confianza del 95%
)

print("🔧 Modelo Prophet creado")

# 2. Entrenar el modelo con datos históricos
print("🚀 Entrenando modelo...")
model.fit(df)
print("✅ Modelo entrenado exitosamente")

## 🔮 Haciendo Predicciones

Ahora vamos a predecir las ventas para los próximos 6 meses (180 días):

In [ ]:
# 3. Crear fechas futuras para hacer predicciones
future_days = 180  # Predecir 6 meses hacia adelante
future = model.make_future_dataframe(periods=future_days)

print(f"📅 Creadas fechas desde {future['ds'].min().date()} hasta {future['ds'].max().date()}")
print(f"📊 Total de fechas: {len(future)} ({len(df)} históricas + {future_days} futuras)")

# 4. Hacer las predicciones
print("\n🔮 Generando predicciones...")
forecast = model.predict(future)
print("✅ Predicciones completadas")

# Ver las últimas predicciones
print("\n🔮 Últimas 10 predicciones:")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10))

### 📊 Visualizando las Predicciones

In [ ]:
# Gráfico principal de Prophet
fig1 = model.plot(forecast, figsize=(15, 8))
plt.title('Predicción de Ventas con Prophet', fontsize=16)
plt.xlabel('Fecha')
plt.ylabel('Ventas ($)')
plt.legend(['Datos históricos', 'Predicción', 'Intervalo de confianza'])
plt.grid(True, alpha=0.3)
plt.show()

print("📖 ¿Cómo leer este gráfico?")
print("🔵 Puntos azules: datos históricos reales")
print("📈 Línea azul: predicción del modelo")
print("🔵 Área sombreada: intervalo de confianza (95%)")
print("📍 La línea vertical marca donde terminan los datos históricos")

## 🔍 Descomponiendo los Componentes

Una de las mejores características de Prophet es que nos muestra **qué componentes** influyen en las predicciones:

In [ ]:
# Mostrar los componentes de la descomposición
fig2 = model.plot_components(forecast, figsize=(15, 10))
plt.tight_layout()
plt.show()

print("📊 ¿Qué nos dice cada gráfico?")
print("")
print("📈 TENDENCIA (arriba):")
print("   - Muestra la dirección general de las ventas")
print("   - ¿Las ventas suben, bajan o se mantienen estables?")
print("")
print("🗓️ ESTACIONALIDAD ANUAL (medio):")
print("   - Patrones que se repiten cada año")
print("   - ¿En qué meses del año se vende más?")
print("")
print("📅 ESTACIONALIDAD SEMANAL (abajo):")
print("   - Patrones que se repiten cada semana")
print("   - ¿Qué días de la semana se vende más?")

## 📊 Análisis de los Resultados

In [ ]:
# Analizar las predicciones futuras
future_predictions = forecast.tail(future_days)
current_avg = df['y'].mean()
future_avg = future_predictions['yhat'].mean()

print("=== RESUMEN DE PREDICCIONES ===")
print(f"📊 Ventas promedio actuales: ${current_avg:.0f}")
print(f"🔮 Ventas promedio predichas (próximos 6 meses): ${future_avg:.0f}")
print(f"📈 Cambio esperado: {((future_avg - current_avg) / current_avg * 100):+.1f}%")
print(f"💰 Venta más alta predicha: ${future_predictions['yhat'].max():.0f}")
print(f"📉 Venta más baja predicha: ${future_predictions['yhat'].min():.0f}")

# Encontrar los mejores días de la semana para vender
print("\n=== MEJORES DÍAS PARA VENDER ===")
weekly_effect = forecast[['ds', 'weekly']].copy()
weekly_effect['day_name'] = weekly_effect['ds'].dt.day_name()
weekly_avg = weekly_effect.groupby('day_name')['weekly'].mean().sort_values(ascending=False)

dias_es = {'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles',
           'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'}

for i, (day, effect) in enumerate(weekly_avg.items(), 1):
    print(f"{i}. {dias_es[day]}: {effect:+.0f} ventas respecto al promedio")

## 💡 Aplicaciones Prácticas de Prophet

### 🛒 **E-commerce y Retail**
- Planificar inventario basado en demanda esperada
- Decidir cuándo hacer promociones
- Estimar ingresos futuros

### 🏢 **Recursos Humanos**
- Predecir cuántos empleados necesitas cada mes
- Planificar vacaciones y ausencias
- Optimizar horarios de trabajo

### 💻 **Tecnología**
- Predecir tráfico web para dimensionar servidores
- Estimar uso de recursos (CPU, memoria)
- Planificar mantenimientos en períodos de bajo uso

### 💰 **Finanzas**
- Proyectar flujo de caja
- Estimar ingresos trimestrales
- Predecir gastos estacionales

## 🚀 Ventajas y Limitaciones de Prophet

### ✅ **Ventajas**
- **Fácil de usar**: No necesitas ser experto en estadística
- **Robusto**: Maneja datos faltantes automáticamente
- **Interpretable**: Te explica qué está pasando
- **Flexible**: Puedes agregar días festivos y eventos especiales
- **Rápido**: Entrena en segundos, no horas

### ⚠️ **Limitaciones**
- **Necesita historia**: Mínimo 1 año de datos para buenas predicciones
- **Cambios abruptos**: No predice bien eventos únicos (como COVID-19)
- **Datos complejos**: Para relaciones muy complejas, otros modelos pueden ser mejores
- **Dependencias**: No considera variables externas automáticamente

## 🎯 Consejos para Usar Prophet Efectivamente

### 📊 **Preparación de Datos**
1. **Frecuencia regular**: Datos diarios, semanales, o mensuales
2. **Sin gaps grandes**: Máximo 1-2 días faltantes
3. **Outliers identificados**: Marca eventos especiales

### 🔧 **Configuración del Modelo**
1. **Ajusta estacionalidades**: Solo activa las que necesites
2. **Agrega festivos**: Días especiales de tu país/industria
3. **Usa regresores**: Variables externas que influyen

### 🎯 **Validación**
1. **Divide tus datos**: Entrena con 80%, prueba con 20%
2. **Validación cruzada**: Prophet tiene herramientas integradas
3. **Monitorea regularmente**: Reentrenar cada mes o trimestre

### 📈 **Interpretación**
1. **Revisa componentes**: ¿Tienen sentido los patrones?
2. **Intervalos de confianza**: Usa el rango, no solo el punto medio
3. **Combina con conocimiento del negocio**: Los datos + tu experiencia = mejores decisiones

## 🎓 Resumen y Próximos Pasos

### 📚 **Lo que aprendimos**
- Prophet es una herramienta poderosa y fácil para pronósticos
- Maneja automáticamente tendencias y estacionalidades
- Proporciona intervalos de confianza útiles
- Es interpretable y práctico para negocios

### 🚀 **Próximos pasos sugeridos**
1. **Practica con tus propios datos**: ¿Tienes datos históricos de tu trabajo/proyecto?
2. **Experimenta con parámetros**: Cambia la estacionalidad, intervalos, etc.
3. **Agrega días festivos**: Usa `holidays` para eventos especiales de tu país
4. **Combina con otras herramientas**: Prophet + dashboard = potencia
5. **Aprende validación cruzada**: Para evaluar qué tan buenos son tus pronósticos

### 📖 **Recursos adicionales**
- [Documentación oficial de Prophet](https://facebook.github.io/prophet/)
- [Prophet en Python: Tutorial completo](https://facebook.github.io/prophet/docs/quick_start.html)
- [Casos de uso reales](https://research.facebook.com/blog/2017/02/prophet-forecasting-at-scale/)

---

**¡Ahora tienes una herramienta poderosa para hacer predicciones! 🎯**